## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-16-10-57-48 +0000,bbc,Two mass shootings in South Africa leave 17 dead,https://www.bbc.co.uk/news/articles/c8y5x3n4ky...
1,2026-11-16-10-57-00 +0000,wsj,UniCredit’s Commerzbank Bid Gets EU Approval,https://www.wsj.com/business/deals/unicredits-...
2,2026-11-16-10-56-00 +0000,wsj,XPeng’s Net Loss Widens Amid Subdued Chinese A...,https://www.wsj.com/business/earnings/xpengs-n...
3,2026-11-16-10-54-00 +0000,wsj,Baidu Profit Misses as It Searches for New Gro...,https://www.wsj.com/business/earnings/baidu-pr...
4,2026-11-16-10-51-45 +0000,bbc,Create more jobs for under-25s by cutting Nati...,https://www.bbc.co.uk/news/articles/c6p9dzmj51...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2652/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
97,trump,20
26,new,14
246,nvidia,12
512,ground,10
49,ai,9


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
207,2026-11-15-17-27-08 +0000,cbc,Trump breaks silence on FAA's nationwide groun...,https://www.cbc.ca/news/world/trump-faa-ground...,65
193,2026-11-15-18-18-45 +0000,nypost,Bessent urges Nvidia to look outside the compa...,https://nypost.com/2026/11/15/us-news/treasury...,60
210,2026-11-15-17-10-54 +0000,nypost,Trump blames Biden admin for FAA outage debacl...,https://nypost.com/2026/11/15/us-news/trump-bl...,57
222,2026-11-15-15-37-44 +0000,wapo,Trump blames Biden for FAA problems in first c...,https://www.washingtonpost.com/politics/2026/1...,53
38,2026-11-16-10-00-00 +0000,wapo,"Midterm losses, Nvidia scandal weaken Trump as...",https://www.washingtonpost.com/politics/2026/1...,52


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
207,65,2026-11-15-17-27-08 +0000,cbc,Trump breaks silence on FAA's nationwide groun...,https://www.cbc.ca/news/world/trump-faa-ground...
193,60,2026-11-15-18-18-45 +0000,nypost,Bessent urges Nvidia to look outside the compa...,https://nypost.com/2026/11/15/us-news/treasury...
124,44,2026-11-15-23-35-02 +0000,cbc,'I cheated': South Korean admits he used AI to...,https://www.cbc.ca/news/world/south-korea-tax-...
52,34,2026-11-16-09-21-31 +0000,bbc,'You're going to win the war': Harry praises w...,https://www.bbc.co.uk/news/articles/c60evn992j...
181,32,2026-11-15-19-30-51 +0000,chitri,Israel kills a top Hamas commander as Jordan s...,https://www.chicagotribune.com/2026/11/15/isra...
144,31,2026-11-15-22-13-00 +0000,wsj,"Miguel Ángel Félix Gallardo, Architect of Mexi...",https://www.wsj.com/world/americas/miguel-ange...
23,29,2026-11-16-10-02-11 +0000,nyt,Democrats Will Wield Enormous Power in Stateho...,https://www.nytimes.com/2026/11/16/us/politics...
138,28,2026-11-15-22-33-03 +0000,nypost,Major Red Sea strait falls under full Houthi c...,https://nypost.com/2026/11/15/world-news/major...
206,27,2026-11-15-17-27-57 +0000,nypost,Dem Sen. Chris Van Hollen boasts he has ‘very ...,https://nypost.com/2026/11/15/us-news/van-holl...
72,25,2026-11-16-06-01-33 +0000,bbc,'I'd throw up in the car park': How school anx...,https://www.bbc.co.uk/news/articles/cm62pl6lgl...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
